In [16]:
import pandas as pd
import numpy as np
from pyMyriad import *

In [17]:
df = pd.DataFrame({
  "id": np.arange(1000),
  "Gender": np.random.choice(["M", "F"], 1000),
  "Age": np.random.randint(18, 70, 1000),
  "Income": np.random.normal(50000, 15000, 1000),
  "Country": np.random.choice(["Benin", "Albania"], 1000),
  "A": np.random.normal(0, 1, 1000),
  "B": np.random.normal(0, 10, 1000)
})

# Basic usage
The analysis tree is constructed by successive split and analyis adding nodes to the tree.
- `split_*` methods add split nodes, dividing or filtering the dataframe.
- `analyze_*` methods add an analysis node that and block further splitting at the node where it was added.
- `summarize_*` methods add an analysis node but won't block further splitting at the node where it was added.

Split and analysis can be specified either as sting of functions.
**Note: use `df` to refer to the dataframe available at the node.**

The resulting analyis tree can be printed to visualize the analysis plan

In [18]:
mfun = lambda df: np.mean(df.Income)
nfun = lambda df: np.std(df.Income)
benin_fun =  lambda df: df.Country == 'Benin'
age_40 = lambda df: df.Age > 40
age_60 = lambda df: df.Age > 60

atree = AnalysisTree()\
  .split_by("df.Gender")\
  .summarize_by(m = mfun, n = nfun)\
  .split_by(label = "Benin Y/N", expr = benin_fun)\
  .split_by(label = "Age Group", age40 = age_40, age60 = age_60)\
  .analyze_by(m = mfun, n = nfun)

print(atree)

Analysis Tree
  └- Split Node df.Gender: [df.Gender]
    └- Analysis Node: 
        m: <function>
        n: <function>
    └- Split Node Benin Y/N: [<function>]
      └- Split Node Age Group: [age40: <function> -- age60: <function>]
        └- Analysis Node: 
            m: <function>
            n: <function>



The `run` method applies the analysis plan on a dataframe and returns a `DataTree`.

In [19]:
res = atree.run(df)
print(res)

Data Tree
  Split: df.Gender
    └- F
        analysis: 
        └- m: 49598.92017276236
        └- n: 14622.462392162724
      Split: Benin Y/N
        └- False
          Split: Age Group
            └- age40
                analysis: 
                └- m: 47381.77277033281
                └- n: 15686.205807207029
            └- age60
                analysis: 
                └- m: 43126.047833839104
                └- n: 14713.048153157164
        └- True
          Split: Age Group
            └- age40
                analysis: 
                └- m: 49376.74945998778
                └- n: 13138.798212665914
            └- age60
                analysis: 
                └- m: 48984.02519337092
                └- n: 12545.782579671832
    └- M
        analysis: 
        └- m: 50535.54064172956
        └- n: 14966.256760672219
      Split: Benin Y/N
        └- False
          Split: Age Group
            └- age40
                analysis: 
                └- m: 51655.62315172879
   

In [ ]:
from pyMyriad.plots import forest_plot
import importlib
ff = forest_plot(res, "m", "n")
display(ff)

statistics,_id,split,type,path_pivot,pivot_lvl,pivot_split,label,y_label,NaN,m,n,y
0,0,NaN,root,root,,,,Overall,None,NaN,NaN,1.0
1,1,df.Gender,split,root >> df.Gender,,,,df.Gender,None,NaN,NaN,2.0
2,3,NaN,analysis,root >> df.Gender >> F >> analysis,,,,F,NaN,49598.920173,14622.462392,3.0
3,4,Benin Y/N,split,root >> df.Gender >> F >> Benin Y/N,,,,Benin Y/N,None,NaN,NaN,4.0
4,5,NaN,level,root >> df.Gender >> F >> Benin Y/N >> False,,,,False,None,NaN,NaN,5.0
5,6,Age Group,split,root >> df.Gender >> F >> Benin Y/N >> False >...,,,,Age Group,None,NaN,NaN,6.0
6,8,NaN,analysis,root >> df.Gender >> F >> Benin Y/N >> False >...,,,,age40,NaN,47381.77277,15686.205807,7.0
7,10,NaN,analysis,root >> df.Gender >> F >> Benin Y/N >> False >...,,,,age60,NaN,43126.047834,14713.048153,8.0
8,11,NaN,level,root >> df.Gender >> F >> Benin Y/N >> True,,,,True,None,NaN,NaN,9.0
9,12,Age Group,split,root >> df.Gender >> F >> Benin Y/N >> True >>...,,,,Age Group,None,NaN,NaN,10.0
